# Darkroom — run everything on a free Colab GPU

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`.

Run the cells **top to bottom** with the ▶ button (or `Shift+Enter`). Each cell says what it does and how long it takes.
Edit only the cell marked **EDIT ME**.

## 1. Check the GPU  ·  ~5 s
You should see a Tesla T4 (or better). If you see an error, the runtime is not set to GPU — see the note above.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. EDIT ME — your team details

In [ ]:
TEAM_ID = "T01"                                   # your team ID from the organisers
USNS    = "1RV23CS001,1RV23CS002,1RV23CS003"      # every member's USN, comma-separated, no spaces
QUICK   = True                                    # True ≈ 30-45 min on a T4; False = full run (≈2 h on a T4)

## 3. Get the code and install  ·  ~3 min

In [ ]:
%cd /content
!rm -rf prachi-darkroom
!git clone -q https://github.com/PrachiAgarwal-coder/prachi-darkroom.git
%cd /content/prachi-darkroom/server
!pip install -q -r requirements.txt
print("Installed.")

## 4. Download the evaluation data (MS-COCO 2017 val)  ·  ~3-5 min
Downloads the official caption file once (~240 MB), then 300 images. Nothing to do by hand.

In [ ]:
!python ../eval/download_coco.py --n 300 --out ../data/coco

## 5. Run the evaluation  ·  QUICK ≈ 30-45 min, full ≈ 2 h
The first run also downloads the models (~5 GB). It prints progress tier by tier. **Keep this tab open.**

In [ ]:
args = "--n 100 --n-baseline 50 --n-ablation 20" if QUICK else ""
!python ../eval/evaluate.py --team "$TEAM_ID" --usns "$USNS" --coco ../data/coco $args

## 6. Look at the results

In [ ]:
from IPython.display import Markdown, Image, display
display(Image("../eval/results/quality_vs_cost.png"))
display(Markdown(open(f"../eval/results/eval_P3_{TEAM_ID}.md").read()))

## 7. Download the results  ·  then upload them to GitHub
This downloads `eval_results.zip`. Unzip it on your laptop and upload the **contents** into the repo's `eval/results/` folder on GitHub (Add file → Upload files).

In [ ]:
!cd ../eval && zip -qr /content/eval_results.zip results -x "results/images/*"
from google.colab import files
files.download("/content/eval_results.zip")

---
## Optional A — live demo server from this notebook (backup if Modal fails)
Starts the model server and gives you a public `https://….trycloudflare.com` URL. It stays alive only while this tab is open.
In the app, click the connection pill and paste `wss://<that-url>/ws`.

In [ ]:
import subprocess, time, re
subprocess.Popen("cd /content/prachi-darkroom/server && nohup uvicorn darkroom.server:app --host 0.0.0.0 --port 8000 > /content/server.log 2>&1 &", shell=True)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
subprocess.Popen("nohup /content/cloudflared tunnel --url http://localhost:8000 > /content/tunnel.log 2>&1 &", shell=True)
for _ in range(60):
    time.sleep(5)
    log = open("/content/tunnel.log").read() if __import__("os").path.exists("/content/tunnel.log") else ""
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
    if m: break
print("Public URL:", m.group(0) if m else "not ready yet - rerun this cell")
print("WebSocket for the app:", m.group(0).replace("https://", "wss://") + "/ws" if m else "")
print("Models take 3-6 min to load; check progress with the next cell.")

In [ ]:
!tail -5 /content/server.log

## Optional B — deploy the permanent server to Modal from here
Run, open the link it prints, log in, then run the deploy cell. The URL it prints is your permanent model server.

In [ ]:
!pip install -q modal && modal token new

In [ ]:
%cd /content/prachi-darkroom
!modal deploy deploy/modal_app.py